## Pytorch minimal workshop

### How to load data
First, import crucial dependencies

In [ ]:

import numpy as np
import pandas as pd

import torch

# If you want to run PyTorch on GPU, you need to run a different torch setup 
# (now defined as additional dependency, to be activated by uv sync --extra cu126 
# This is highly dependent on your architecture!! Let me know if that is something of interest!

# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))




Load a dataset using pandas. Pandas is still quite convenient for working with data, but ultimately, pytorch needs tensors (which are, just like DataFrames, built on top of numpy arrays). Any datasplits should also be done before transforming the dataframes into tensors.

In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

print(y.shape)
print(X.head())

There are different ways to convert a dataframe into a tensor:

1) Directly from dataframe, via the `values` property.

In [ ]:
X_tensor = torch.tensor(X.values)
X_tensor

2. Via `torch.tensor` but via numpy array. Likewise, numpy arrays can be directly converted into tensors.

In [ ]:
X_tensor = torch.tensor(X.to_numpy(), dtype=torch.float32) # datatype conversion optional
X_tensor

3. Using `DataLoader` for large datasets (data larger than allocated memory)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

X_tensor = torch.tensor(X.to_numpy())
y_tensor = torch.tensor(y.to_numpy())

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch in dataloader:
    print(batch)

4. Use a Custom Dataset Class for more flexibility and cleaner code:

In [ ]:

from torch.utils.data import Dataset, DataLoader

class MyDataFromDF(Dataset):
    def __init__(self, X, y):
        # features provided as Dataframes converted
        self.X = torch.tensor(    
            X.to_numpy(), 
            # dtype=torch.float32() # optionally include a datatype
            ) 
        
        # targets provided as Dataframes
        self.y = torch.tensor( 
            y.to_numpy(),
            # dtype=torch.float32() # optionally include a datatype
            )            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)


In [ ]:
# Wrap custom dataset in Dataloader

dataset = MyDataFromDF(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# for batch in loader:
#     print(batch)

In [ ]:
# How to use the loader in training:
for batch_X, batch_y in loader:
    pred = model(batch_X)
    loss = criterion(pred, batch_y)
    ...


Datatypes need to be handled by torch (integer, float, Boolean) - compatible dtypes will be converted into a common format (check with `print(tensor.dtype)`). A datatype can be specified in (`dtype=torch.float32`) - see some examples above.

Strings, categorical values are incompatible and need to be encoded into numerical values.

## Build a NN for Regression
For convenience and better comparison, the Diabetes dataset as available in scikit-learn is used here again.

Train-test split on original data:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Convert to tensors - you could also use any other method from above! Note that here there was no DataLoader used as wrapper, as the dataset is small.

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

A fully connected NN is defined as model class. Note the separation of the initialisation and the forward pass class functions (essentially separating the learning part from the activation functions).

In [ ]:
import torch.nn as nn

class DiabetesNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        self.fc2 = nn.Linear(hidden_size, hidden_size)  # one hidden layer, try to add another one
        self.fc3 = nn.Linear(hidden_size, output_size)  # output layer
    
    # Good practice: activation functions specified in forward pass - separated from the learning parts
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        # no activation function for the output layer (because it is a regresion model!)
        return x

Define hyperparameters:

In [ ]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 20
output_size = 1
learning_rate = 0.05

# Number of training iterations
num_epochs = 100

Introduce loss function (``criterion``) and gradient optimiser (``optimizer``):

In [ ]:
import torch.optim as optim
model = DiabetesNN(input_size, hidden_size, output_size)

criterion = nn.MSELoss()  # loss function defined;

optimizer = optim.Adam(model.parameters(), learning_rate) # gradient descent method based on average and squares of gradient

In [ ]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using SDG

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Evaluate the model on the test set to termine the generalisation for the regression model.

In [ ]:
# Evaluate the model
mse_loss = nn.MSELoss()

model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    mse = mse_loss(y_pred, y_test).item()
    print("MSE:", mse)
